In [0]:
###Document Structure

from langchain_core.documents import Document

In [0]:
%sh
pip install langchain_community
pip install pymupdf

Read PDF File From Here

In [0]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

dir_loader=DirectoryLoader(
    "./Data/PDF_Files",
    glob="**/*.pdf", ## Pattern to match files  
    loader_cls= PyMuPDFLoader, ##loader class to use
    show_progress=False

)

pdf_documents = dir_loader.load()

print(f"Total pages loaded: {len(pdf_documents)}")
pdf_documents

In [0]:
# Check total documents loaded
print(f"Total documents (pages) loaded: {len(pdf_documents)}")
print(f"\nUnique PDF files loaded:")

# Get unique file sources
file_sources = set()
for doc in pdf_documents:
    file_sources.add(doc.metadata.get('source', 'Unknown'))

for i, source in enumerate(sorted(file_sources), 1):
    file_name = source.split('/')[-1]
    page_count = sum(1 for doc in pdf_documents if doc.metadata.get('source') == source)
    print(f"{i}. {file_name}: {page_count} pages")

print(f"\nTotal unique PDF files: {len(file_sources)}")

In [0]:
type(pdf_documents[0])

PDF Loader Part Starting From Here

In [0]:
%pip install langchain-community langchain 'langchain-protocol<0.0.18' chromadb
dbutils.library.restartPython()

In [0]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [0]:
%sh 
pip install chromadb
pip install sentence_transformers
pip install pypdf

In [0]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""

    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)
            print(f"✓ Loaded {len(documents)} pages")

        except Exception as e:
            print(f"✗ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents


# Update with your workspace path
all_pdf_documents = process_all_pdfs(
    "/Workspace/Users/draxop7536@gmail.com/RAG_1/Data/PDF_Files"
)

print(all_pdf_documents[:2])

Split PDF into smaller chunks

In [0]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [0]:
chunks=split_documents(all_pdf_documents)
chunks

In [0]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

Embadding

In [0]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

In [0]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "./Data/Vector_Store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

In [0]:
chunks

In [0]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

## Store in the vector database
# Use a fresh directory path to avoid ChromaDB lock/corruption issues
import uuid
import shutil

# Create a unique directory name to ensure clean state
vector_store_path = f"/Workspace/Users/draxop7536@gmail.com/RAG_1/Data/Vector_Store_{uuid.uuid4().hex[:8]}"

vectorstore = VectorStore(persist_directory=vector_store_path)
vectorstore.add_documents(chunks,embeddings)



# ### Convert the text to embeddings
# texts=[doc.page_content for doc in chunks]

# ## Generate the Embeddings

# embeddings=embedding_manager.generate_embeddings(texts)

# ##store int he vector dtaabase
# vectorstore.add_documents(chunks,embeddings)

Temporary data retrival testing from vectorstore

In [0]:
# Query
query_text = "In which year this new constitution was adopted?"

# Generate query embedding using the same embedding model
query_embedding = embedding_manager.generate_embeddings([query_text])[0]

results = vectorstore.collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=2
)

# Format the output
print(f"Query: {query_text}")
print("=" * 80)
print()

if results['documents'] and len(results['documents'][0]) > 0:
    for i, (doc, metadata, distance) in enumerate(zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    )):
        print(f"Result {i+1}:")
        print(f"Source: {metadata.get('source_file', 'N/A')}")
        print(f"Page: {metadata.get('page', 'N/A') + 1}")
        print(f"Relevance Score: {1 - distance:.4f}")
        print()
        print("Content:")
        print(doc)
        print()
        print("-" * 80)
        print()
else:
    print("No results found.")

Retrival Logic

In [0]:
from typing import List, Dict, Any

class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store, embedding_manager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: Search query
            top_k: Number of documents to retrieve

        Returns:
            List of retrieved documents
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}")

        try:
            # Generate query embedding
            query_embedding = self.embedding_manager.generate_embeddings([query])[0]

            # Query ChromaDB
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            retrieved_docs = []

            if results["documents"] and len(results["documents"][0]) > 0:

                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for rank, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances),
                    start=1
                ):

                    retrieved_docs.append({
                        "id": doc_id,
                        "content": document,
                        "metadata": metadata,
                        "distance": distance,
                        "rank": rank
                    })

                    print(f"Rank: {rank} | Distance: {distance:.4f}")

                print(f"Retrieved {len(retrieved_docs)} documents")

            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [0]:
rag_retriever

In [0]:
# Query the RAG system (recreate retriever to use the proper RAGRetriever class)
query_text = "In which year this new constitution was adopted?"
rag_retriever = RAGRetriever(vectorstore, embedding_manager)
results = rag_retriever.retrieve(query_text, top_k=5)


# Format the output nicely
print("\n" + "=" * 80)
print(f"Query: {query_text}")
print("=" * 80)
print()

if results:
    for result in results:
        print(f"Rank {result['rank']}:")
        print(f"Source: {result['metadata'].get('source_file', 'N/A')}")
        print(f"Page: {result['metadata'].get('page', 'N/A') + 1}")
        print(f"Relevance Score: {1 - result['distance']:.4f}")
        print()
        print("Content:")
        print(result['content'])
        print()
        print("-" * 80)
        print()
else:
    print("No results found.")

Integrating VectorDB Context pipeline With LLM Output

In [0]:
import os
from dotenv import load_dotenv
load_dotenv()

print(os.getenv("GROQ_API_KEY"))

Same as in GITHUB

In [0]:
%pip install -q langchain-groq langchain
dbutils.library.restartPython()

In [0]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [0]:
class GroqLLM:
    def __init__(self, model_name: str = "gemma2-9b-it", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"

In [0]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

In [0]:
rag_retriever.retrieve("In which year this new constitution was adopted?")

In [0]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [0]:
answer=rag_simple("In which year this new constitution was adopted?",rag_retriever,llm)
print(answer)

In [0]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Filter by minimum score (distance-based: lower distance = higher similarity)
    filtered_results = [doc for doc in results if (1 - doc['distance']) >= min_score]
    
    if not filtered_results:
        return {'answer': 'No relevant context found meeting the minimum score threshold.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in filtered_results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': 1 - doc['distance'],
        'preview': doc['content'][:300] + '...'
    } for doc in filtered_results]
    confidence = max([1 - doc['distance'] for doc in filtered_results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("In which year this new constitution was adopted?", rag_retriever, llm, top_k=3, min_score=0.05, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

In [0]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            # Filter by minimum score (distance-based: lower distance = higher similarity)
            filtered_results = [doc for doc in results if (1 - doc['distance']) >= min_score]
            
            if not filtered_results:
                answer = "No relevant context found meeting the minimum score threshold."
                sources = []
                context = ""
            else:
                context = "\n\n".join([doc['content'] for doc in filtered_results])
                sources = [{
                    'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                    'page': doc['metadata'].get('page', 'unknown'),
                    'score': 1 - doc['distance'],
                    'preview': doc['content'][:120] + '...'
                } for doc in filtered_results]
                results = filtered_results
                
                # Generate answer only if we have context
                # Streaming answer simulation
                prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
                if stream:
                    print("Streaming answer:")
                    for i in range(0, len(prompt), 80):
                        print(prompt[i:i+80], end='', flush=True)
                        time.sleep(0.05)
                    print()
                response = self.llm.invoke([prompt])
                answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("The Constituent Assembly of India originally had how many members?", top_k=3, min_score=0.05, stream=False, summarize=True)

print("\n" + "="*80)
print("ADVANCED RAG PIPELINE RESULTS")
print("="*80)
print("\nQuestion:", result['question'])
print("\nAnswer:", result['answer'])
print("\nSources:")
for i, src in enumerate(result['sources'], 1):
    print(f"  [{i}] {src['source']} (page {src['page']}) - Score: {src['score']:.4f}")
print("\nSummary:", result['summary'])
print("\nQuery History Count:", len(result['history']))